In [0]:
# ─────────────────────────────────────────────────────────────
# Cell 1 — CONFIG
# ─────────────────────────────────────────────────────────────
CATALOG       = "cricket"
BRONZE_SCHEMA = "bronze"
VOLUME        = "landing"
FEED_URL      = "https://cricsheet.org/downloads/recently_added_2_json.zip"

VOLUME_PATH  = f"/Volumes/{CATALOG}/{BRONZE_SCHEMA}/{VOLUME}"
ZIP_PATH     = f"{VOLUME_PATH}/recently_added_2.zip"
JSON_DIR     = f"{VOLUME_PATH}/json"
BRONZE_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.raw_matches"

In [0]:
# ─────────────────────────────────────────────────────────────
# Cell 2 — Volume + landing dir (idempotent)
# ─────────────────────────────────────────────────────────────
import os
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}.{VOLUME}")
os.makedirs(JSON_DIR, exist_ok=True)
print("Landing ready:", VOLUME_PATH)

In [0]:
# ─────────────────────────────────────────────────────────────
# Cell 3 — Download the delta zip to the Volume
# If serverless egress blocks this, download locally and upload the
# zip into the Volume via the UI, then skip to Cell 4.
# ─────────────────────────────────────────────────────────────
import requests
resp = requests.get(FEED_URL, timeout=120)
resp.raise_for_status()
with open(ZIP_PATH, "wb") as f:
    f.write(resp.content)
print(f"Downloaded {len(resp.content):,} bytes -> {ZIP_PATH}")

In [0]:
# Cell 4 — clear landing dir first, then unzip only THIS run's files
import zipfile, shutil, os

shutil.rmtree(JSON_DIR, ignore_errors=True)   # wipe stale files from prior runs
os.makedirs(JSON_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH) as z:
    members = [m for m in z.namelist() if m.endswith(".json") and not m.startswith("_")]
    z.extractall(JSON_DIR, members=members)

files = [f for f in os.listdir(JSON_DIR) if f.endswith(".json")]
print(f"Extracted {len(files)} match files. Sample: {files[:3]}")

In [0]:
# ─────────────────────────────────────────────────────────────
# Cell 5 (VARIANT) — Land raw as queryable semi-structured data
# One row per match: match_id + revision + lineage as typed columns,
# the full document as VARIANT (faithful, queryable, schema-absorbing).
# ─────────────────────────────────────────────────────────────
from pyspark.sql import functions as F
from pyspark.sql.window import Window

raw = spark.read.text(f"{JSON_DIR}/*.json", wholetext=True)   # column: value

stamped = (
    raw
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("match_id",
                F.regexp_extract("_metadata.file_path", r"([0-9]+)\.json$", 1))
    .withColumn("data",  F.expr("parse_json(value)"))                 # -> VARIANT
    .withColumn("data_version", F.expr("data:meta:data_version::string"))
    .withColumn("revision",
                F.coalesce(F.expr("data:meta:revision::int"), F.lit(1)))
    .withColumn("_ingested_at", F.current_timestamp())
    .select("match_id", "revision", "data_version",
            "data", "_source_file", "_ingested_at")
)

# de-dup within the batch: highest revision per match_id (handles the 2-day overlap)
w = Window.partitionBy("match_id").orderBy(F.col("revision").desc())
stamped = stamped.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")

print("Rows to load:", stamped.count())
stamped.select("match_id", "revision", "data_version").show(5, truncate=False)

In [0]:
# ─────────────────────────────────────────────────────────────
# Cell 6 — MERGE with revision guard (unchanged; VARIANT rides along)
# ─────────────────────────────────────────────────────────────
from delta.tables import DeltaTable

if not spark.catalog.tableExists(BRONZE_TABLE):
    (stamped.write.format("delta")
        .clusterBy("match_id")
        .saveAsTable(BRONZE_TABLE))
    print("Created", BRONZE_TABLE)
else:
    tgt = DeltaTable.forName(spark, BRONZE_TABLE)
    (tgt.alias("t")
        .merge(stamped.alias("s"), "t.match_id = s.match_id")
        .whenMatchedUpdateAll(condition="s.revision > t.revision")
        .whenNotMatchedInsertAll()
        .execute())
    print("Merged into", BRONZE_TABLE)

In [0]:
# ─────────────────────────────────────────────────────────────
# Cell 7 — Verify (query the VARIANT directly — this is the payoff)
# ─────────────────────────────────────────────────────────────
spark.sql(f"""
  SELECT
    data:info:match_type::string  AS match_type,
    data:info:balls_per_over::int AS balls_per_over,
    count(*)                       AS matches
  FROM {BRONZE_TABLE}
  GROUP BY ALL
  ORDER BY matches DESC
""").show()